<a href="https://colab.research.google.com/github/Tritiksha20/Machine-Learning-College/blob/main/9_Esemble_learning_models_for_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.metrics import accuracy_score, classification_report

In [8]:
df = pd.read_csv("/content/breast_cancer.csv")
X = df.drop(columns=['target', 'diagnosis'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
results = {}

In [9]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=50, random_state=42
)
bagging.fit(X_train_scaled, y_train)
y_pred = bagging.predict(X_test_scaled)
results['Bagging (Decision Trees)'] = accuracy_score(y_test, y_pred)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)
results['Random Forest'] = accuracy_score(y_test, y_pred)

In [10]:
adaboost = AdaBoostClassifier(n_estimators=100, random_state=42)
adaboost.fit(X_train_scaled, y_train)
y_pred = adaboost.predict(X_test_scaled)
results['AdaBoost'] = accuracy_score(y_test, y_pred)

gboost = GradientBoostingClassifier(n_estimators=100, random_state=42)
gboost.fit(X_train_scaled, y_train)
y_pred = gboost.predict(X_test_scaled)
results['Gradient Boosting'] = accuracy_score(y_test, y_pred)

In [11]:
clf1 = LogisticRegression(max_iter=5000, random_state=42)
clf2 = SVC(kernel='rbf', probability=True, random_state=42)
clf3 = DecisionTreeClassifier(random_state=42)

voting = VotingClassifier(
    estimators=[('lr', clf1), ('svc', clf2), ('dt', clf3)],
    voting='soft'
)
voting.fit(X_train_scaled, y_train)
y_pred = voting.predict(X_test_scaled)
results['Voting Classifier (soft)'] = accuracy_score(y_test, y_pred)

In [12]:
print("=" * 60)
print("ENSEMBLE LEARNING MODEL COMPARISON")
print("=" * 60)
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{name:35s}: {acc:.4f}")

best_model_name = max(results, key=results.get)
print(f"\nBest performing ensemble model: {best_model_name} ({results[best_model_name]:.4f})")

print("\n" + "=" * 60)
print(f"Detailed report for Random Forest (feature importances - top 10):")
print("=" * 60)
importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(10))

print("\nClassification report for best model (Voting Classifier):")
print(classification_report(y_test, voting.predict(X_test_scaled),
                             target_names=['malignant', 'benign']))

ENSEMBLE LEARNING MODEL COMPARISON
Voting Classifier (soft)           : 0.9720
AdaBoost                           : 0.9650
Random Forest                      : 0.9580
Gradient Boosting                  : 0.9580
Bagging (Decision Trees)           : 0.9510

Best performing ensemble model: Voting Classifier (soft) (0.9720)

Detailed report for Random Forest (feature importances - top 10):
worst area              0.149674
worst concave points    0.127189
mean concave points     0.104650
worst radius            0.086963
worst perimeter         0.080299
mean perimeter          0.080037
mean concavity          0.055420
mean radius             0.053665
mean area               0.044062
area error              0.024557
dtype: float64

Classification report for best model (Voting Classifier):
              precision    recall  f1-score   support

   malignant       0.98      0.94      0.96        53
      benign       0.97      0.99      0.98        90

    accuracy                           0.97